# 01 – Data Ingestion
**Bluestock Fintech Mutual Fund Analytics Platform**

This notebook loads all 10 raw CSVs, profiles each dataset, validates columns, and saves initial load summaries.

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

BASE  = Path("..").resolve()
RAW   = BASE / "data" / "raw"
PROC  = BASE / "data" / "processed"
PROC.mkdir(exist_ok=True)

FILES = {
    "fund_master":           "01_fund_master.csv",
    "nav_history":           "02_nav_history.csv",
    "aum_by_fund_house":     "03_aum_by_fund_house.csv",
    "monthly_sip_inflows":   "04_monthly_sip_inflows.csv",
    "category_inflows":      "05_category_inflows.csv",
    "industry_folio_count":  "06_industry_folio_count.csv",
    "scheme_performance":    "07_scheme_performance.csv",
    "investor_transactions": "08_investor_transactions.csv",
    "portfolio_holdings":    "09_portfolio_holdings.csv",
    "benchmark_indices":     "10_benchmark_indices.csv",
}

datasets = {}
for name, fname in FILES.items():
    df = pd.read_csv(RAW / fname, low_memory=False)
    datasets[name] = df
    print(f"✅ {name:30s} | {df.shape[0]:>7,} rows × {df.shape[1]:>3} cols")

✅ fund_master                    |      40 rows ×  15 cols
✅ nav_history                    |  46,000 rows ×   3 cols
✅ aum_by_fund_house              |      90 rows ×   5 cols
✅ monthly_sip_inflows            |      48 rows ×   6 cols
✅ category_inflows               |     144 rows ×   3 cols
✅ industry_folio_count           |      21 rows ×   6 cols
✅ scheme_performance             |      40 rows ×  19 cols
✅ investor_transactions          |  32,778 rows ×  13 cols
✅ portfolio_holdings             |     322 rows ×   8 cols
✅ benchmark_indices              |   8,050 rows ×   3 cols


## Dataset Profiles

In [2]:
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  {name.upper()}")
    print(f"{'='*60}")
    print(df.dtypes.to_string())
    print(f"\nMissing values:")
    missing = df.isnull().sum()
    print(missing[missing > 0].to_string() if missing.any() else "  None")


  FUND_MASTER
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object

Missing values:
  None

  NAV_HISTORY
amfi_code      int64
date          object
nav          float64

Missing values:
  None

  AUM_BY_FUND_HOUSE
date               object
fund_house         object
aum_lakh_crore    float64
aum_crore           int64
num_schemes         int64

Missing values:
  None

  MONTHLY_SIP_INFLOWS
month                         object
sip_inflow_crore               int64
active_sip_accounts_crore    float64
new_sip_accounts_lakh        float64
sip_aum_lakh_crore           float64
yoy_growth_pct            

## Row Count Summary

In [3]:
summary = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": df.shape[1],
     "null_cells": df.isnull().sum().sum(),
     "duplicate_rows": df.duplicated().sum()}
    for name, df in datasets.items()
])
print(summary.to_string(index=False))
summary.to_csv(PROC / "ingestion_summary.csv", index=False)
print("\nIngestion summary saved.")

                table  rows  columns  null_cells  duplicate_rows
          fund_master    40       15           0               0
          nav_history 46000        3           0               0
    aum_by_fund_house    90        5           0               0
  monthly_sip_inflows    48        6          12               0
     category_inflows   144        3           0               0
 industry_folio_count    21        6           0               0
   scheme_performance    40       19           0               0
investor_transactions 32778       13           0               0
   portfolio_holdings   322        8           0               0
    benchmark_indices  8050        3           0               0

Ingestion summary saved.


## Sample Records

In [4]:
for name, df in list(datasets.items())[:3]:
    print(f"\n--- {name} (first 3 rows) ---")
    print(df.head(3).to_string())


--- fund_master (first 3 rows) ---
   amfi_code       fund_house                                 scheme_name category sub_category     plan launch_date             benchmark  expense_ratio_pct  exit_load_pct  min_sip_amount  min_lumpsum_amount   fund_manager risk_category sebi_category_code
0     119551  SBI Mutual Fund   SBI Bluechip Fund - Regular Plan - Growth   Equity    Large Cap  Regular  2006-02-14         NIFTY 100 TRI               1.54            1.0             500                1000  Sohini Andani      Moderate               EC01
1     119552  SBI Mutual Fund    SBI Bluechip Fund - Direct Plan - Growth   Equity    Large Cap   Direct  2013-01-01         NIFTY 100 TRI               0.66            1.0             500                1000  Sohini Andani      Moderate               EC01
2     119598  SBI Mutual Fund  SBI Small Cap Fund - Regular Plan - Growth   Equity    Small Cap  Regular  2009-09-09  BSE 250 SmallCap TRI               1.43            1.0             500     

In [6]:
print("\nData Ingestion Notebook Complete")
print(f"All {len(datasets)} datasets loaded successfully.")


Data Ingestion Notebook Complete
All 10 datasets loaded successfully.
